# Llama 3.1 8B access probe\nNo GPU, model weights, research data, or training are used. This checks only the Kaggle `HF_TOKEN` secret and access to the pinned model config.\nEnable Internet and select the existing `HF_TOKEN` secret under Add-ons → Secrets before running. The notebook never prints the token.\n

In [ ]:
# Hash-bound probe; it reads no weights or research data.
import hashlib
source = '#!/usr/bin/env python3\n"""Verify Kaggle HF secret and gated Llama config access without GPU/weights."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport sys\nfrom pathlib import Path\n\nMODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"\nREVISION = "0e9e39f249a16976918f6564b8830bc894c89659"\nOUTPUT = Path("/kaggle/working/r1_8b_access_probe.json")\n\n\ndef get_token() -> str:\n    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")\n    if token:\n        return token\n    from kaggle_secrets import UserSecretsClient\n\n    value = UserSecretsClient().get_secret("HF_TOKEN")\n    if not value:\n        raise RuntimeError("HF_TOKEN secret is empty")\n    return value\n\n\ndef main() -> int:\n    OUTPUT.parent.mkdir(parents=True, exist_ok=True)\n    receipt = {\n        "status": "STARTED",\n        "model_id": MODEL_ID,\n        "expected_revision": REVISION,\n        "scope": "SECRET_AND_CONFIG_ACCESS_ONLY",\n        "gpu_required": False,\n        "weights_downloaded": False,\n        "research_data_loaded": False,\n        "training_started": False,\n    }\n    try:\n        from huggingface_hub import HfApi, hf_hub_download\n\n        token = get_token()\n        info = HfApi(token=token).model_info(MODEL_ID, revision=REVISION, token=token)\n        if info.sha != REVISION:\n            raise RuntimeError(f"Resolved revision mismatch: {info.sha}")\n        config_path = Path(\n            hf_hub_download(\n                repo_id=MODEL_ID,\n                filename="config.json",\n                revision=REVISION,\n                token=token,\n            )\n        )\n        config = json.loads(config_path.read_text(encoding="utf-8"))\n        if not isinstance(config, dict) or config.get("model_type") != "llama":\n            raise RuntimeError("Downloaded config is not a Llama model config")\n        receipt.update(\n            {\n                "status": "PASS_ACCESS_ONLY",\n                "resolved_revision": info.sha,\n                "config_sha256": hashlib.sha256(config_path.read_bytes()).hexdigest(),\n                "config_model_type": config.get("model_type"),\n                "token_value_logged": False,\n            }\n        )\n    except Exception as exc:\n        receipt.update(\n            {\n                "status": "FAIL_ACCESS_ONLY",\n                "error_type": type(exc).__name__,\n                "error": str(exc)[:1000],\n                "token_value_logged": False,\n            }\n        )\n        OUTPUT.write_text(json.dumps(receipt, indent=2) + "\\n", encoding="utf-8")\n        print(json.dumps(receipt, indent=2))\n        raise\n    OUTPUT.write_text(json.dumps(receipt, indent=2) + "\\n", encoding="utf-8")\n    print(json.dumps(receipt, indent=2))\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n'
expected = '6da1a1a8035803d594d131c22555595c9e561acef432493f3cb2d0bf71c7643c'
actual = hashlib.sha256(source.encode('utf-8')).hexdigest()
if actual != expected:
    raise RuntimeError(f'Probe source hash mismatch: {actual}')
namespace = {'__name__': 'kaggle_hf_access_probe'}
exec(compile(source, 'probe.py', 'exec'), namespace)
namespace['main']()
